[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxischa/datacamp_test/blob/main/bloc4_ml/exercices/seance4_exercices.ipynb)

# Séance 4.4 — Segmenter sans étiquette — quatre clients, quatre traitements

**Exercices** · durée : 2h (≈50 min de cours, ≈50 min d'exercices)

> ⚠️ **Avant de taper quoi que ce soit :** *Fichier → Enregistrer une copie dans Drive*. Sinon votre travail sera perdu en fermant l'onglet.
>
> 📱 Sur tablette, faites d'abord les réglages de [Bien démarrer](https://github.com/maxischa/datacamp_test/blob/main/ressources/setup_tablette.md).

## Objectifs

À la fin de cette séance, vous saurez :

- distinguer un problème supervisé d'un problème non supervisé
- expliquer pourquoi il faut standardiser avant de mesurer une distance
- appliquer `KMeans` et choisir le nombre de groupes
- donner un nom et un chiffre d'affaires à chaque segment obtenu
- transformer une segmentation en plan d'action budgété

## Comment ça marche

La feuille compte **deux parties**, à faire dans l'ordre.

**Partie 1 — l'échauffement.** Le code est déjà écrit, il ne reste que les `____` à
remplir. Chaque exercice se termine par une cellule de **vérification** qui vous dit
immédiatement si votre réponse est bonne.

**Partie 2 — les questions.** Une question, une cellule **vide** : à vous d'écrire le
code entier. Il n'y a pas de vérification automatique — on les corrige ensemble en
séance, et la correction est publiée après.

> ⚠️ Si une vérification de la partie 1 affiche `NameError`, c'est que la cellule
au-dessus n'a pas été exécutée, ou qu'il y reste un `____`. Complétez-la, relancez-la,
puis relancez la vérification.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

# Affichage adapte aux petits ecrans
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 80)

# Les donnees sont lues directement depuis le web : rien a telecharger
BASE = "https://raw.githubusercontent.com/maxischa/datacamp_test/main/bloc4_ml/data/"

In [ ]:
def verifier(nom, condition, indice=""):
    """Affiche un retour immediat sans interrompre le notebook."""
    print("OK   -", nom) if condition else print("A REVOIR -", nom, ":", indice)

Chargement des données utilisées dans toute la feuille :

In [ ]:
cli = pd.read_csv(BASE + "clients_rfm.csv")   ## une ligne = un client

variables = ["recence", "freq", "montant"]   ## le trio RFM

# log1p ecrase les valeurs extremes, StandardScaler egalise les echelles
Xs = StandardScaler().fit_transform(np.log1p(cli[variables]))
print(cli.shape, "| variables mises a l'echelle")

---

# Partie 1 — L'échauffement

Le code est déjà écrit : il ne reste que les `____` à remplir. Allez vite, l'essentiel
de la séance est dans la partie 2.

### Exercice 1 — Le clustering qui rate

> **Votre mission :**
> - Lancer un `KMeans` à 4 groupes sur les colonnes **brutes**, sans standardiser → `brut`.
> - Mettre la taille du plus gros groupe dans `plus_gros` et celle du plus petit dans `plus_petit`.

In [ ]:
brut = KMeans(n_clusters=4, n_init=10, random_state=42).fit(cli[____])
tailles = pd.Series(brut.labels_).value_counts()

plus_gros = tailles.max()
plus_petit = tailles.____()
print(plus_gros, "clients dans le plus gros groupe,", plus_petit, "dans le plus petit")

In [ ]:
verifier("1a - plus gros groupe", plus_gros == 401, "value_counts() puis max()")
verifier("1b - plus petit groupe", plus_petit == 2, "la methode est min()")

### Exercice 2 — La courbe du coude

> **Votre mission :**
> - Pour `k` de 2 à 8, relever l'inertie du `KMeans` → `inerties` (une `Series` indexée par k).
> - La tracer. Mettre l'inertie à `k` = 4 dans `inertie4`, arrondie à 1 décimale.

In [ ]:
inerties = pd.Series(
    {k: KMeans(n_clusters=k, n_init=10, random_state=42).fit(Xs).____
     for k in range(2, 9)})

inerties.plot(marker="o", figsize=(7, 4))
plt.title("Courbe du coude")
plt.show()

inertie4 = round(inerties[4], 1)
print(inertie4)

In [ ]:
verifier("2 - inertie a k=4", abs(inertie4 - 453.7) < 5,
         "l'attribut s'appelle inertia_, avec un underscore final")

### Exercice 3 — La silhouette

> **Votre mission :**
> - Calculer le score de silhouette pour `k` de 2 à 6 → `silhouettes`.
> - Mettre le `k` qui la maximise dans `k_silhouette`.
> - Est-ce celui qu'on va retenir ?

In [ ]:
silhouettes = pd.Series(
    {k: silhouette_score(Xs, KMeans(n_clusters=k, n_init=10, random_state=42).fit(Xs).labels_)
     for k in range(2, 7)})

k_silhouette = silhouettes.____()
print(silhouettes.round(3))
print("meilleur k selon la silhouette :", k_silhouette)

In [ ]:
verifier("3 - k preferee par la silhouette", k_silhouette == 2,
         "idxmax() donne l'indice du maximum")

### Exercice 4 — Former les quatre groupes

> **Votre mission :**
> - Ajuster un `KMeans` à 4 groupes sur `Xs` → `km`, et ranger les étiquettes dans une colonne `groupe` de `cli`.
> - Mettre la taille du plus petit groupe dans `taille_min`.

In [ ]:
km = KMeans(n_clusters=4, n_init=10, random_state=42).fit(Xs)
cli["groupe"] = km.____

taille_min = cli["groupe"].value_counts().min()
print(cli["groupe"].value_counts().sort_index())

In [ ]:
verifier("4 - plus petit groupe", taille_min == 69,
         "l'attribut des etiquettes s'appelle labels_")

### Exercice 5 — Le tableau des personas

> **Votre mission :**
> - Construire `profils` : par groupe, l'effectif, les **médianes** de récence, fréquence et montant, et la somme des montants.
> - Ajouter une colonne `part_ca` en % du CA total.

In [ ]:
profils = cli.groupby("groupe").agg(
    n=("client_id", "size"),
    recence=("recence", "____"),
    freq=("freq", "median"),
    montant=("montant", "median"),
    ca=("montant", "____"),
)
profils["part_ca"] = (100 * profils["ca"] / cli["montant"].sum()).round(1)
print(profils.round(1))

In [ ]:
verifier("5 - part du CA du plus gros segment", abs(profils["part_ca"].max() - 60.7) < 1,
         "median() pour les profils, sum() pour le CA")

### Exercice 6 — Nommer les groupes

> **Votre mission :**
> - Identifier les groupes par leurs propriétés, sans se fier au numéro (il change d'une exécution à l'autre).
> - Le groupe des **dormants** est celui dont la récence médiane est la plus élevée → `g_dormants`.
> - Celui des **champions** est celui dont la fréquence médiane est la plus élevée → `g_champions`.

In [ ]:
g_dormants = profils["recence"].____()
g_champions = profils["____"].idxmax()

print("dormants : groupe", g_dormants, "| champions : groupe", g_champions)

In [ ]:
verifier("6a - segment dormant", profils.loc[g_dormants, "recence"] > 150,
         "la recence la plus elevee : idxmax()")
verifier("6b - segment champion", profils.loc[g_champions, "freq"] >= 9,
         "la frequence la plus elevee")

### Exercice 7 — Le nuage coloré

> **Votre mission :**
> - Tracer récence en abscisse et fréquence en ordonnée, un point par client, une couleur par groupe.
> - Mettre la fréquence en **échelle logarithmique** — sinon les champions écrasent la figure.
> - Mettre le nombre de clients tracés dans `nb_points`.

In [ ]:
for g in sorted(cli["groupe"].unique()):
    part = cli.query("groupe == @g")
    plt.scatter(part["recence"], part["freq"], alpha=0.6, label=f"groupe {g}")

plt.____("log")
plt.xlabel("jours depuis le dernier achat")
plt.ylabel("nombre de commandes (echelle log)")
plt.legend()
plt.show()

nb_points = len(cli)

In [ ]:
verifier("7 - tous les clients sont traces", nb_points == 472,
         "la methode s'appelle plt.yscale")

### Exercice 8 — Question de synthèse

> **Votre mission :**
> - Chiffrer l'arbitrage : le CA déjà dépensé par les dormants → `ca_dormants` (2 décimales), et leur nombre → `nb_dormants`.
> - Puis le nombre de clients du segment « nouveaux » — celui dont la fréquence est la plus basse **et** la récence faible → `nb_nouveaux`.
> - Où mettriez-vous 5 000 € de relance ? Répondez en commentaire.

In [ ]:
dormants = cli.query("groupe == @g_dormants")
ca_dormants = round(dormants["montant"].____(), 2)
nb_dormants = len(dormants)

recents = profils.query("recence < 100")
g_nouveaux = recents["freq"].idxmin()
nb_nouveaux = int(profils.loc[g_nouveaux, "n"])
print(nb_dormants, "dormants (", ca_dormants, "euros ) |", nb_nouveaux, "nouveaux")

In [ ]:
verifier("8a - nombre de dormants", nb_dormants == 149, "sum() sur les montants du segment")
verifier("8b - CA des dormants", abs(ca_dormants - 74683.3) < 100, "somme des montants")
verifier("8c - nombre de nouveaux", nb_nouveaux == 98, "la frequence la plus basse parmi les recents")

---

# Partie 2 — Les questions

Ici, plus de trous : **la cellule sous chaque question est vide**, et c'est à vous
d'écrire le code en entier. C'est exactement ce qu'on vous demandera pour le projet
final, et ce que fait un analyste devant un fichier qu'il découvre.

Certaines questions utilisent une commande que le cours n'a pas montrée. Quand c'est le
cas, l'énoncé vous la donne — savoir se servir d'une commande qu'on vient de lire fait
partie du métier.

> 💡 Pas de vérification automatique dans cette partie. Affichez systématiquement votre
> résultat, et demandez-vous s'il est **plausible** avant de passer à la suite : c'est
> le seul contrôle dont vous disposerez en entreprise.

### Question 9 — La segmentation est-elle stable ?

> **Votre mission :**
> - Relancer `KMeans` avec cinq `random_state` différents et comparer les tailles de groupes obtenues.
> - Les segments sont-ils les mêmes ? Que faudrait-il vérifier avant de bâtir une campagne dessus ?

### Question 10 — Segmenter sans le montant

> **Votre mission :**
> - Refaire la segmentation sur `recence` et `freq` seulement.
> - Comparer les profils obtenus à ceux du cours.
> - Le montant apportait-il quelque chose que les deux autres variables ne disaient pas ?

### Question 11 — Le client le plus atypique

> **Votre mission :**
> - Pour chaque client, calculer sa distance au centre de son propre groupe.
> - Afficher les cinq plus éloignés. Que sont-ils ?
> - *Nouveau :* `km.transform(Xs)` donne la distance de chaque point à **chaque** centre.

### Question 12 — Segmenter les produits

> **Votre mission :**
> - Charger `produits_profil.csv` : 1 263 références avec leur profil de vente.
> - Appliquer la même méthode — `log1p`, standardisation, `KMeans` à 4 groupes — sur `nb_cmd`, `prix`, `pays` et `part_q4`.
> - Décrire les quatre familles obtenues.

### Question 13 — La famille saisonnière

> **Votre mission :**
> - Parmi les familles de produits, identifier celle dont la part du chiffre d'affaires réalisée au dernier trimestre est la plus élevée.
> - Combien de références contient-elle, et quel CA représente-t-elle ?
> - Quelle décision d'approvisionnement en tirez-vous ?

### Question 14 — Croiser les deux segmentations

> **Votre mission :**
> - Question ouverte, et plus difficile : les champions achètent-ils les mêmes familles de produits que les autres segments ?
> - Il faut repartir de `ventes.csv` du bloc 2 pour relier clients et produits. L'URL est donnée ci-dessous.
> - *Indice :* joindre les ventes aux groupes clients, puis aux familles produits, puis croiser.

### Question 15 — Le livrable du bloc

> **Votre mission :**
> - Rédigez la note qui clôt le bloc 4 : *« comment devons-nous traiter nos clients ? »*
> - Contrainte : quatre segments nommés, chacun avec son effectif, sa part du CA et **une** action ; plus une phrase sur ce que ces données ne permettent pas d'affirmer.
> - Calculez d'abord le tableau dont vous avez besoin.